# Baseline Clustering Analysis

Python/Zerve notebook version of `Clustering/Basline_Analysis.R`.

This notebook focuses on the complete-case baseline dataset and reproduces the main clustering workflow:

- numeric-only k-means diagnostics
- average-linkage hierarchical clustering
- Gaussian mixture clustering
- mixed-type clustering with a k-prototypes style representation
- Gower-distance hierarchical clustering
- pairwise ARI comparison across available methods

R-only methods from the original script, such as fuzzy k-means, PD clustering, and skew-t mixture modeling, are not included here because they depend on specialized R packages.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.metrics import adjusted_rand_score, pairwise_distances, silhouette_score
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")
RANDOM_STATE = 123

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "Clustering" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "Datasets" / "V2"
OUTPUT_DIR = PROJECT_ROOT / "Clustering" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT

## 1. Load Baseline Dataset

In [ ]:
baseline_path = DATA_DIR / "data_complete_baseline.csv"
data_baseline = pd.read_csv(baseline_path)

print(data_baseline.shape)
data_baseline.head()

In [ ]:
# Match the R script: fill remaining baseline income missing values with the modal/middle category.
data_baseline["income"] = data_baseline["income"].fillna("$40,000 - $59,999")

data_baseline.isna().sum().sort_values(ascending=False).head(15)

## 2. Numeric-Only K-Means

This section mirrors the first part of `Basline_Analysis.R`: select numeric survey/behavior variables, standardize them, inspect elbow and silhouette diagnostics, then fit a final k-means model.

In [ ]:
NUMERIC_COLS = [
    "WorkTimeInSeconds...9",
    "multiplier",
    "amount",
    "WorkTimeInSeconds...17",
    "Q3_1.On.a.scale.of.0.to.100..how.would.you.describe.your.political.views.",
    "Q8_1.On.a.scale.of.0.to.100..how.would.you.describe.your.religious.orientation.",
]

missing_cols = [col for col in NUMERIC_COLS if col not in data_baseline.columns]
if missing_cols:
    raise ValueError(f"Missing expected numeric columns: {missing_cols}")

data_numeric = data_baseline[NUMERIC_COLS].apply(pd.to_numeric, errors="coerce").dropna()
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data_numeric)

print(data_numeric.shape)
data_numeric.describe().round(2)

In [ ]:
k_values = range(1, 11)
wss = []

for k in k_values:
    km = KMeans(n_clusters=k, n_init=25, random_state=RANDOM_STATE)
    km.fit(data_scaled)
    wss.append(km.inertia_)

elbow_df = pd.DataFrame({"k": list(k_values), "wss": wss})
display(elbow_df)

fig, ax = plt.subplots(figsize=(7, 4))
sns.lineplot(data=elbow_df, x="k", y="wss", marker="o", ax=ax)
ax.set_title("Elbow Method for K-Means")
ax.set_xlabel("Number of clusters (k)")
ax.set_ylabel("Total within-cluster sum of squares")
plt.show()

In [ ]:
sil_rows = []
for k in range(2, 11):
    labels = KMeans(n_clusters=k, n_init=25, random_state=RANDOM_STATE).fit_predict(data_scaled)
    sil_rows.append({"k": k, "silhouette": silhouette_score(data_scaled, labels)})

sil_df = pd.DataFrame(sil_rows)
display(sil_df)

fig, ax = plt.subplots(figsize=(7, 4))
sns.lineplot(data=sil_df, x="k", y="silhouette", marker="o", ax=ax)
ax.set_title("Silhouette Method for K-Means")
ax.set_xlabel("Number of clusters (k)")
ax.set_ylabel("Average silhouette score")
plt.show()

In [ ]:
kmeans_2 = KMeans(n_clusters=2, n_init=25, random_state=RANDOM_STATE)
kmeans_labels = kmeans_2.fit_predict(data_scaled)

centers_original_scale = pd.DataFrame(
    scaler.inverse_transform(kmeans_2.cluster_centers_),
    columns=NUMERIC_COLS,
    index=["cluster_0", "cluster_1"]
)

print(pd.Series(kmeans_labels).value_counts().sort_index())
display(centers_original_scale.round(2))

## 3. Hierarchical Clustering on Numeric Variables

In [ ]:
linkage_matrix = linkage(data_scaled, method="average", metric="euclidean")
hc_labels = fcluster(linkage_matrix, t=2, criterion="maxclust")

fig, ax = plt.subplots(figsize=(10, 4))
dendrogram(linkage_matrix, no_labels=True, color_threshold=None, ax=ax)
ax.set_title("Average-Linkage Hierarchical Clustering")
ax.set_xlabel("Observations")
ax.set_ylabel("Distance")
plt.show()

ari_kmeans_hc = adjusted_rand_score(kmeans_labels, hc_labels)
print(f"ARI: k-means vs hierarchical = {ari_kmeans_hc:.4f}")

## 4. Gaussian Mixture Model

This is the Python counterpart of the `mclust` section. Scikit-learn's `GaussianMixture` does not choose covariance model exactly the same way as `mclust`, so the numeric values may differ from R, but the comparison logic is the same.

In [ ]:
gmm_rows = []
for k in range(1, 11):
    gmm = GaussianMixture(n_components=k, covariance_type="full", random_state=RANDOM_STATE, n_init=5)
    gmm.fit(data_scaled)
    gmm_rows.append({"k": k, "bic": gmm.bic(data_scaled), "aic": gmm.aic(data_scaled)})

gmm_df = pd.DataFrame(gmm_rows)
display(gmm_df)

fig, ax = plt.subplots(figsize=(7, 4))
sns.lineplot(data=gmm_df, x="k", y="bic", marker="o", label="BIC", ax=ax)
sns.lineplot(data=gmm_df, x="k", y="aic", marker="o", label="AIC", ax=ax)
ax.set_title("Gaussian Mixture Model Selection")
ax.set_xlabel("Number of components")
plt.show()

In [ ]:
comparison_rows = []
gmm_labels_by_k = {}

for k in [2, 3]:
    km_labels = KMeans(n_clusters=k, n_init=25, random_state=RANDOM_STATE).fit_predict(data_scaled)
    gmm_labels = GaussianMixture(n_components=k, covariance_type="full", random_state=RANDOM_STATE, n_init=5).fit_predict(data_scaled)
    gmm_labels_by_k[k] = gmm_labels
    comparison_rows.append({"k": k, "ARI_kmeans_vs_gmm": adjusted_rand_score(km_labels, gmm_labels)})

pd.DataFrame(comparison_rows).round(4)

## 5. Mixed-Type Baseline Clustering

The R script uses `clustMixType::kproto` for mixed numeric/categorical data and `cluster::daisy(metric = "gower")` for mixed-distance hierarchical clustering.

In this baseline notebook, mixed k-prototypes is represented with scaled numeric variables plus one-hot encoded categorical variables. The dedicated k-prototypes notebook uses the closer Python package `kmodes.KPrototypes`.

In [ ]:
STATE_TO_REGION = {
    **dict.fromkeys(["CT", "ME", "MA", "NH", "RI", "VT", "NJ", "NY", "PA"], "Northeast"),
    **dict.fromkeys(["IL", "IN", "IA", "KS", "MI", "MN", "MO", "NE", "ND", "OH", "SD", "WI"], "Midwest"),
    **dict.fromkeys(["AL", "AR", "DE", "DC", "FL", "GA", "KY", "LA", "MD", "MS", "NC", "OK", "SC", "TN", "TX", "VA", "WV"], "South"),
    **dict.fromkeys(["AK", "AZ", "CA", "CO", "HI", "ID", "MT", "NV", "NM", "OR", "UT", "WA", "WY", "0"], "West"),
}

AGE_LEVELS = ["18-29", "30-39", "40-49", "50-59", "60-69", "70 or over"]
INCOME_LEVELS = ["Under $20,000", "$20,000 - $39,999", "$40,000 - $59,999", "$60,000 - $79,999", "$80,000 - $99,999", "Over $100,000"]
ATTENDANCE_LEVELS = ["Never", "Seldom", "A few times a year", "Once or twice a month", "Once a week", "More than once a week"]

STATE_COL = "Q9.What.State.do.you.live.in."
VOTE_COL = "Q5.In.the.2016.Presidential.election..who.did.you.vote.for."
PARTY_COL = "Q7.Do.you.consider.yourself.a."
ATTEND_COL = "Q8.Aside.from.weddings.and.funerals..how.often.do.you.attend.religious.services."
POL_COL = "Q3_1.On.a.scale.of.0.to.100..how.would.you.describe.your.political.views."
REL_COL = "Q8_1.On.a.scale.of.0.to.100..how.would.you.describe.your.religious.orientation."

MIXED_NUMERIC_COLS = NUMERIC_COLS
MIXED_CATEGORICAL_COLS = ["age", "income", "gender", VOTE_COL, STATE_COL, PARTY_COL, ATTEND_COL, "batch"]

def prepare_mixed_baseline(df):
    out = df.copy()
    out[STATE_COL] = out[STATE_COL].astype(str).str.strip().str.upper().map(STATE_TO_REGION)
    for col in MIXED_NUMERIC_COLS:
        out[col] = pd.to_numeric(out[col], errors="coerce")
    cols = MIXED_NUMERIC_COLS + MIXED_CATEGORICAL_COLS
    out = out[cols].dropna().copy()
    return out

mixed_df = prepare_mixed_baseline(data_baseline)
print(mixed_df.shape)
mixed_df.head()

In [ ]:
encoded = pd.get_dummies(mixed_df, columns=MIXED_CATEGORICAL_COLS, drop_first=False)
encoded[MIXED_NUMERIC_COLS] = StandardScaler().fit_transform(encoded[MIXED_NUMERIC_COLS])

mixed_encoded_labels = KMeans(n_clusters=2, n_init=25, random_state=RANDOM_STATE).fit_predict(encoded)

print(pd.Series(mixed_encoded_labels).value_counts().sort_index())

In [ ]:
def gower_distance_matrix(df, numeric_cols, categorical_cols):
    numeric = df[numeric_cols].astype(float).copy()
    ranges = numeric.max() - numeric.min()
    ranges = ranges.replace(0, 1)
    numeric_scaled = (numeric - numeric.min()) / ranges
    numeric_dist = pairwise_distances(numeric_scaled, metric="manhattan") / len(numeric_cols)

    cat = df[categorical_cols].astype(str)
    cat_dist = np.zeros((len(df), len(df)))
    for col in categorical_cols:
        values = cat[col].to_numpy()
        cat_dist += (values[:, None] != values[None, :]).astype(float)
    cat_dist = cat_dist / len(categorical_cols)

    return (numeric_dist * len(numeric_cols) + cat_dist * len(categorical_cols)) / (len(numeric_cols) + len(categorical_cols))

gower_dist = gower_distance_matrix(mixed_df, MIXED_NUMERIC_COLS, MIXED_CATEGORICAL_COLS)
gower_hc = AgglomerativeClustering(n_clusters=2, metric="precomputed", linkage="average")
gower_hc_labels = gower_hc.fit_predict(gower_dist)

ari_mixed = adjusted_rand_score(mixed_encoded_labels, gower_hc_labels)
print(f"ARI: mixed encoded k-means vs Gower hierarchical = {ari_mixed:.4f}")

## 6. Pairwise ARI Matrix

In [ ]:
# Align all methods to the rows used in mixed_df.
aligned_numeric = mixed_df[NUMERIC_COLS].astype(float)
aligned_scaled = StandardScaler().fit_transform(aligned_numeric)

cluster_labels = {
    "kmeans": KMeans(n_clusters=2, n_init=25, random_state=RANDOM_STATE).fit_predict(aligned_scaled),
    "hclust_avg": AgglomerativeClustering(n_clusters=2, linkage="average").fit_predict(aligned_scaled),
    "gmm": GaussianMixture(n_components=2, covariance_type="full", random_state=RANDOM_STATE, n_init=5).fit_predict(aligned_scaled),
    "mixed_encoded_kmeans": mixed_encoded_labels,
    "gower_hclust": gower_hc_labels,
}

methods = list(cluster_labels)
ari_matrix = pd.DataFrame(index=methods, columns=methods, dtype=float)
for a in methods:
    for b in methods:
        ari_matrix.loc[a, b] = adjusted_rand_score(cluster_labels[a], cluster_labels[b])

display(ari_matrix.round(4))

cluster_sizes = pd.DataFrame({name: pd.Series(labels).value_counts().sort_index() for name, labels in cluster_labels.items()})
display(cluster_sizes)

## 7. Baseline Cluster Profile Plot

This figure profiles each clustering method by comparing cluster-level averages. Cluster labels are oriented so that **Cluster 2 has the higher average religious orientation** within each method. The left panel shows `Cluster 2 average - Cluster 1 average`; the right panel shows the raw cluster averages by method.


In [ ]:
# Build a baseline cluster profile figure similar to the report slide.
# Cluster labels are relabeled so Cluster 2 is the more religiously oriented cluster.
profile_df = mixed_df.copy()
profile_df["Age"] = pd.Categorical(profile_df["age"], categories=AGE_LEVELS, ordered=True).codes + 1
profile_df["Amt"] = pd.to_numeric(profile_df["amount"], errors="coerce")
profile_df["Attend"] = pd.Categorical(profile_df[ATTEND_COL], categories=ATTENDANCE_LEVELS, ordered=True).codes + 1
profile_df["Pol"] = pd.to_numeric(profile_df[POL_COL], errors="coerce")
profile_df["Rel"] = pd.to_numeric(profile_df[REL_COL], errors="coerce")
profile_df["Vote"] = pd.Categorical(profile_df[VOTE_COL].astype(str)).codes + 1

PROFILE_VARIABLES = ["Age", "Amt", "Attend", "Pol", "Rel", "Vote"]
METHOD_LABELS = {
    "kmeans": "kmeans",
    "hclust_avg": "hclust_numeric",
    "gmm": "gmm",
    "mixed_encoded_kmeans": "mixed_encoded",
    "gower_hclust": "hclust_gower",
}

profile_long_parts = []
for method, labels in cluster_labels.items():
    tmp = profile_df[PROFILE_VARIABLES].copy()
    raw = pd.Series(labels, index=tmp.index)

    rel_means = tmp.assign(raw_cluster=raw).groupby("raw_cluster")["Rel"].mean()
    high_religion_cluster = rel_means.idxmax()
    tmp["Cluster"] = np.where(raw == high_religion_cluster, "Cluster 2", "Cluster 1")
    tmp["Method"] = METHOD_LABELS.get(method, method)
    profile_long_parts.append(tmp)

profile_long = pd.concat(profile_long_parts, ignore_index=True)
profile_average = (
    profile_long
    .groupby(["Method", "Cluster"], as_index=False)[PROFILE_VARIABLES]
    .mean()
    .melt(id_vars=["Method", "Cluster"], var_name="Variable", value_name="Average")
)

profile_wide = profile_average.pivot_table(
    index=["Method", "Variable"],
    columns="Cluster",
    values="Average",
).reset_index()
profile_wide["Difference"] = profile_wide["Cluster 2"] - profile_wide["Cluster 1"]
profile_wide["Direction"] = np.where(profile_wide["Difference"] >= 0, "Cluster 2 higher", "Cluster 1 higher")

method_order = list(METHOD_LABELS.values())
variable_order = PROFILE_VARIABLES
cluster_order = ["Cluster 1", "Cluster 2"]

fig = plt.figure(figsize=(18, 8))
outer = fig.add_gridspec(1, 2, width_ratios=[1.05, 1.35], wspace=0.22)
left_grid = outer[0].subgridspec(2, 3, hspace=0.35, wspace=0.25)
right_grid = outer[1].subgridspec(len(variable_order), 2, hspace=0.35, wspace=0.12)

palette_diff = {"Cluster 2 higher": "#4C9F70", "Cluster 1 higher": "#D2691E"}
cluster_palette = {"Cluster 1": "#D2691E", "Cluster 2": "#4C9F70"}

for idx, variable in enumerate(variable_order):
    ax = fig.add_subplot(left_grid[idx // 3, idx % 3])
    data = profile_wide[profile_wide["Variable"] == variable].copy()
    data["Method"] = pd.Categorical(data["Method"], categories=method_order, ordered=True)
    data = data.sort_values("Method")
    colors = data["Direction"].map(palette_diff)
    ax.barh(data["Method"], data["Difference"], color=colors)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(variable)
    ax.set_xlabel("Cluster 2 average - Cluster 1 average")
    ax.set_ylabel("" if idx % 3 else "Method")
    ax.grid(axis="x", alpha=0.25)

for row, variable in enumerate(variable_order):
    for col, cluster in enumerate(cluster_order):
        ax = fig.add_subplot(right_grid[row, col])
        data = profile_average[
            (profile_average["Variable"] == variable)
            & (profile_average["Cluster"] == cluster)
        ].copy()
        data["Method"] = pd.Categorical(data["Method"], categories=method_order, ordered=True)
        data = data.sort_values("Method")
        ax.bar(data["Method"], data["Average"], color=cluster_palette[cluster])
        ax.set_title(cluster if row == 0 else "")
        ax.set_ylabel(variable)
        ax.tick_params(axis="x", rotation=55, labelsize=8)
        ax.grid(axis="y", alpha=0.25)

fig.suptitle("Baseline Cluster Profile Comparison", fontsize=18, fontweight="bold", y=0.995)
fig.text(
    0.5,
    0.01,
    "Cluster labels are oriented within each method so Cluster 2 has higher average religious orientation.",
    ha="center",
    fontsize=11,
)
plt.tight_layout(rect=[0, 0.04, 1, 0.96])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
plt.savefig(OUTPUT_DIR / "baseline_cluster_profile_comparison.png", dpi=200, bbox_inches="tight")
plt.show()